# 01 — Frame the prediction before training

**Plain-language question:** What decision are we trying to support, and what
may the model know at prediction time?

**Why this matters:** a technically good model can still solve the wrong
problem. We define the question, timing, action, and errors before comparing
algorithms.

**Estimated time:** 35–45 minutes.
**Prerequisite:** lesson 00; you know row, feature, label, probability, and
threshold.


## Preflight

Run this first. It checks the kernel and shows where this lesson reads and
writes course state.


In [ ]:
import importlib.util
import sys

required = ("mlflow", "pandas", "sklearn", "aai_local_classification")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "This notebook is using the wrong Python kernel. Close this Jupyter "
        "server, run `make notebook` from examples/local-classification, or "
        "select the 'AAI Local Classification' kernel. Missing: " + ", ".join(missing)
    )

import pandas as pd

from aai_local_classification.learning import study_root
from aai_local_classification.settings import load_settings
from aai_local_classification.tracking import local_paths

settings = load_settings()
root = study_root()
paths = local_paths(root)
print(f"✓ Python {sys.version_info.major}.{sys.version_info.minor}: {sys.executable}")
print("✓ Course imports are available")
print(f"✓ Learner state: {root}")


In [ ]:
from aai_local_classification.contracts import SplitName
from aai_local_classification.data import load_split
from aai_local_classification.workflow import ensure_prepared

ensure_prepared(settings, root)
train = load_split(settings, SplitName.TRAIN, paths.data_root)
print(f"✓ Loaded {len(train):,} training rows")


### What you should see

The environment checks and `✓ Loaded 2,160 training rows`. If this notebook was
opened directly, the setup cell safely created the deterministic prerequisite
data rather than training a hidden model.


## Put the question on a timeline

```text
facts available now                 answer learned later
monthly snapshot ── predict ── optional review ── churn within 30 days?
        ↑ prediction time                         ↑ prediction horizon
```

### Words introduced

| Word | Plain meaning | Here |
|---|---|---|
| prediction time | The instant inputs must be available | monthly snapshot |
| prediction horizon | How far ahead the answer covers | the next 30 days |
| positive class | The event represented by `1` | churn |

The fictional action is to send a positive prediction to a human retention
review. The model does not contact a customer and does not make an autonomous
decision.


Look at one example vertically. `churned_30d` is shown so we can learn from
historical training data, but it cannot be an input when making the prediction.


In [ ]:
example = train.loc[0, [*settings.features.model_columns, "churned_30d"]]
example.to_frame("value")


### How to interpret what you should notice

Numbers such as `monthly_fee` and categories such as `contract_type` describe
the account. The label is the later answer. A data contract prevents code from
quietly moving the answer—or facts created after the answer—into the feature
list.


## Which columns play which role?

A **numeric feature** has meaningful numeric magnitude. A **categorical
feature** names a group. A context field helps identify or order a row without
being learned directly. A forbidden field is unavailable or unsafe at
prediction time.


In [ ]:
roles = pd.DataFrame(
    [
        ("tenure_months", "numeric feature", "months already subscribed"),
        ("monthly_fee", "numeric feature", "current monthly charge"),
        ("contract_type", "categorical feature", "contract arrangement"),
        ("autopay", "categorical feature", "automatic payment enabled"),
        ("account_id", "context", "row identifier, not a learned signal"),
        ("snapshot_date", "context", "defines prediction time and split"),
        ("churned_30d", "target", "answer during the next 30 days"),
        ("cancellation_reason", "forbidden", "only known after cancellation"),
    ],
    columns=["column", "role", "meaning"],
)
roles


The full reviewed feature lists live in `configs/project.yaml`. Raw IDs are
usually poor model inputs. Dates can support legitimate prediction-time
features (for example, month of year), but this course deliberately uses the
raw snapshot date only for lineage and splitting.


## How uncommon is the positive class?

**Before you run this:** if churn is uncommon, which count do you expect to be
larger: `0` or `1`?


In [ ]:
label_counts = train.churned_30d.value_counts().sort_index()
label_summary = pd.DataFrame(
    {
        "meaning": ["no churn", "churn"],
        "rows": [label_counts[0], label_counts[1]],
        "share": [label_counts[0] / len(train), label_counts[1] / len(train)],
    },
    index=pd.Index([0, 1], name="label"),
)
label_summary


### What you should see

1,833 non-churn rows and 327 churn rows: about 15.1% are positive. This
**prevalence** matters because a model that always says “no churn” would look
about 84.9% accurate while missing every churn. Lesson 04 makes that failure
visible with a confusion matrix.


## Name the two error types before assigning costs

### Words introduced

| Error | Plain meaning | Fictional consequence |
|---|---|---|
| false positive | predict churn, but no churn occurs | unnecessary review |
| false negative | predict no churn, but churn occurs | missed review opportunity |
| action cost | a teaching weight used to compare those errors | FP = 1, FN = 5 |

The numbers are units for a worked example—not dollars and not researched
business impact. A real team would agree them with decision owners, capacity
owners, risk specialists, and affected users.


In [ ]:
cost_example = pd.DataFrame(
    {
        "false_positives": [20, 0],
        "false_negatives": [0, 20],
    },
    index=["20 unnecessary reviews", "20 missed churns"],
)
cost_example["teaching_cost"] = (
    cost_example.false_positives * settings.selection.false_positive_cost
    + cost_example.false_negatives * settings.selection.false_negative_cost
)
cost_example


### What you should see

Twenty false positives cost 20 teaching units; twenty false negatives cost 100.
This makes recall important and will usually move the chosen threshold below
0.5. It does not mean “predict everyone positive”: reviews still have a cost.

### Misconception check

The target is never a prediction-time input. A column being stored in the same
historical table does not make it available when the real prediction occurs.


### Guided exercise

Classify `cancellation_reason` as `allowed` or `forbidden`, then explain why.
Change the starter value if needed.


In [ ]:
exercise_role = "forbidden"
exercise_reason = "It is only known after the customer has cancelled."
pd.Series({"role": exercise_role, "reason": exercise_reason})


**Self-check:** a feature must exist at the monthly snapshot. If knowing its
value requires waiting for the outcome, it leaks the answer.

<details><summary>Solution explanation</summary>

`cancellation_reason` is forbidden. It describes an event that happens after
the prediction time, so it would make historical scores unrealistically good.
</details>


In [ ]:
# Reference solution — run after your attempt
assert exercise_role == "forbidden"
assert "after" in exercise_reason.lower()
print("✓ The post-outcome field is excluded")


## MLOps bridge

The prediction contract belongs in reviewed configuration and run evidence so
that “good score” cannot silently replace “right decision.” MLflow will later
record the target, positive label, costs, and selection policy beside each run.

## Recap

- We predict churn within 30 days using only facts available at the snapshot.
- The positive class is uncommon, and the two error types have different
  illustrative costs.
- Context, target, and post-outcome fields stay outside the model features.

**Evidence created:** none beyond lesson 00's deterministic data. This lesson
reads the authored contract from `configs/project.yaml`.

**Ready for 02?** You can state the prediction time, target, positive class,
fictional action, and why `cancellation_reason` is forbidden.
